In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from sklearn.metrics import calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors

In [2]:
# =========================
# 1. Load data
# =========================
celltagmulti_path = "/Users/apple/Desktop/KB/data/Cell_tag-Cell_tag_multi_integrated/Seurat_method/cellTag_test_multi_Seurat_clone_id.h5ad"
celltag_path      = "/Users/apple/Desktop/KB/data/Cell_tag-Cell_tag_multi_integrated/Seurat_method/cellTag_train_tag_Seurat.h5ad"

Cell_tagMulti = ad.read_h5ad(celltagmulti_path)
Cell_tag      = ad.read_h5ad(celltag_path)

adata_Cell_Multi_labels = Cell_tagMulti.obs["clone_id"].to_numpy()
adata_Cell_tag_labels   = Cell_tag.obs["clone_id"].to_numpy()

In [3]:

# =========================
# 2. Load ONE embedding
# =========================
embed_path = "/Users/apple/Desktop/KB/data/feat_LCL_2025/Cell_tag-Cell_tag_multi"

# Example: no semi-supervised head
test_emb_path = os.path.join(
    embed_path,
    "feat_celltagMulti_non-semi",
    "test_proj_embed.npy"
)


test_embedding = np.load(test_emb_path)

print("Loaded test embedding:", test_embedding.shape)
print("Number of CellTag labels:", len(adata_Cell_tag_labels))


Loaded test embedding: (6534, 32)
Number of CellTag labels: 6534


In [4]:

# =========================
# 3. Calinski–Harabasz on test
# =========================
def calinski_on_test(test_emb: np.ndarray, labels) -> float:
    """
    Calinski–Harabasz index on labeled test cells.
    Assumes labels align with rows of test_emb.
    """
    codes, _ = pd.factorize(pd.Series(labels), sort=True)
    if np.unique(codes).size < 2:
        raise ValueError("Need at least 2 unique lineages for CH score.")
    return calinski_harabasz_score(test_emb, codes)


# =========================
# 4. Size-normalized KNN enrichment
# =========================
def knn_lineage_enrichment(
    test_emb: np.ndarray,
    labels,
    K: int = 50,
    metric: str = "cosine",
    return_details: bool = False,
):
    """
    For each cell i:
      p_local  = (# same-lineage among K NN) / K
      p_global = (lineage_size_i - 1) / (N - 1)
      enrichment_i = p_local / p_global
    Returns mean & median enrichment (and details if requested).
    """
    N = test_emb.shape[0]
    labels = pd.Series(labels)
    codes, uniques = pd.factorize(labels, sort=True)
    if K >= N:
        K = N - 1

    # KNN over all cells; first neighbor is self
    nn = NearestNeighbors(n_neighbors=K + 1, metric=metric)
    nn.fit(test_emb)
    _, nbrs = nn.kneighbors(test_emb, return_distance=True)
    nbrs = nbrs[:, 1:]  # drop self

    # global lineage sizes
    lineage_sizes = pd.Series(codes).value_counts().to_dict()

    same_in_K = np.zeros(N, dtype=int)
    enrich = np.empty(N, dtype=float)

    for i in range(N):
        yi = codes[i]
        kn_codes = codes[nbrs[i]]
        same = np.sum(kn_codes == yi)
        same_in_K[i] = same

        n_L = lineage_sizes[yi] - 1  # exclude self
        p_global = n_L / (N - 1) if N > 1 else 0.0
        p_local = same / K
        enrich[i] = (p_local / p_global) if p_global > 0 else np.nan

    mean_enrichment = np.nanmean(enrich)
    median_enrichment = np.nanmedian(enrich)

    if not return_details:
        return mean_enrichment, median_enrichment

    per_cell = pd.DataFrame({
        "cell_index": np.arange(N),
        "lineage_code": codes,
        "lineage_label": uniques[codes],
        "same_in_K": same_in_K,
        "enrichment": enrich
    })

    per_lineage = (
        per_cell.groupby("lineage_code")
        .agg(
            n_cells=("cell_index", "size"),
            mean_enrichment=("enrichment", "mean"),
            median_enrichment=("enrichment", "median")
        )
        .reset_index()
    )
    per_lineage["lineage_label"] = uniques[per_lineage["lineage_code"]]

    return mean_enrichment, median_enrichment, per_lineage


In [5]:
# =========================
# 5. Evaluate ONE embedding
# =========================
K_list = [20, 50, 100]
METRIC = "euclidean"   # or "cosine"

assert test_embedding.shape[0] == len(adata_Cell_tag_labels), (
    f"Row mismatch: embedding has {test_embedding.shape[0]} rows, "
    f"but labels have {len(adata_Cell_tag_labels)} entries."
)

ch = calinski_on_test(test_embedding, adata_Cell_tag_labels)

knn_mean_20, knn_median_20 = knn_lineage_enrichment(
    test_embedding, adata_Cell_tag_labels, K=K_list[0], metric=METRIC, return_details=False
)
knn_mean_50, knn_median_50 = knn_lineage_enrichment(
    test_embedding, adata_Cell_tag_labels, K=K_list[1], metric=METRIC, return_details=False
)
knn_mean_100, knn_median_100 = knn_lineage_enrichment(
    test_embedding, adata_Cell_tag_labels, K=K_list[2], metric=METRIC, return_details=False
)

results_df = pd.DataFrame([{
    "embedding_path": test_emb_path,
    "n_cells": test_embedding.shape[0],
    "dim": test_embedding.shape[1],
    "calinski_harabasz": round(ch, 4),
    f"knn_mean_K{K_list[0]}_{METRIC}": round(knn_mean_20, 4),
    f"knn_median_K{K_list[0]}_{METRIC}": round(knn_median_20, 4),
    f"knn_mean_K{K_list[1]}_{METRIC}": round(knn_mean_50, 4),
    f"knn_median_K{K_list[1]}_{METRIC}": round(knn_median_50, 4),
    f"knn_mean_K{K_list[2]}_{METRIC}": round(knn_mean_100, 4),
    f"knn_median_K{K_list[2]}_{METRIC}": round(knn_median_100, 4),
}])

print(results_df.to_string(index=False))

                                                                                                   embedding_path  n_cells  dim  calinski_harabasz  knn_mean_K20_euclidean  knn_median_K20_euclidean  knn_mean_K50_euclidean  knn_median_K50_euclidean  knn_mean_K100_euclidean  knn_median_K100_euclidean
/Users/apple/Desktop/KB/data/feat_LCL_2025/Cell_tag-Cell_tag_multi/feat_celltagMulti_non-semi/test_proj_embed.npy     6534   32             3.1252                  1.9859                    0.7492                  2.0047                    1.0739                   1.9365                     1.1951
